In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, hashlib, subprocess
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
for m in ['config','features_cic','features_ugr16']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# Cell 2 - shared conformal helpers that return COUNTS (n_eval, n_covered) per
# class per draw, so CIC/UGR can be recomputed binomial-ready from saved probs.
# Same randomized APS / Mondrian as nb13/nb17. ALPHA = primary only (0.05), the
# level all three datasets share.
# =============================================================================
ALPHA = config.ALPHA_PRIMARY
def dseed(*p): return int(hashlib.sha256('|'.join(map(str,p)).encode()).hexdigest(),16)%(2**32)
def aps_scores_all(P, rng):
    order=np.argsort(-P,axis=1); sp=np.take_along_axis(P,order,1)
    cum=np.cumsum(sp,1); U=rng.random(len(P))[:,None]
    ss=cum-(1-U)*sp; out=np.empty_like(P); np.put_along_axis(out,order,ss,1); return out
def qhat(ts, alpha):
    n=len(ts)
    return np.inf if n<1 else float(np.quantile(ts, min(np.ceil((n+1)*(1-alpha))/n,1.0), method='higher'))
def cov_counts(eval_scores, y_eval, cal_all, y_cal, alpha, ncls):
    tc=cal_all[np.arange(len(y_cal)),y_cal]
    q={c:qhat(tc[y_cal==c],alpha) for c in range(ncls)}
    ne={}; nc={}
    for c in range(ncls):
        mk=y_eval==c; ne[c]=int(mk.sum())
        nc[c]=int((eval_scores[mk,c]<=q[c]).sum()) if mk.any() else 0
    return ne,nc
R=getattr(config,'N_MATCHED_DRAWS',10)
COLS=['dataset','unit_id','seed','arch','protocol','class','is_focal','n_eval','n_covered',
      'S_cov','S_lab','S_sup','alpha','nominal']
print('helpers ready; pooling at alpha =', ALPHA, '| matched draws =', R)


helpers ready; pooling at alpha = 0.05 | matched draws = 10


In [3]:
# =============================================================================
# Cell 3 - NSL-KDD rows (already count-based). Merge coverage with its per-
# (rung,realization) shift measures; keep alpha=0.05; harmonise to COLS.
# Focal = R2L.
# =============================================================================
ncov = pd.read_csv(config.REPORTS_DIR/'coverage_primary_nslkdd.csv')
nsh  = pd.read_csv(config.REPORTS_DIR/'ladder_shift_measures_nslkdd.csv')
ncov = ncov[ncov['alpha']==ALPHA].copy()
m = ncov.merge(nsh[['rung','realization','S_cov','S_lab','S_sup']], on=['rung','realization'], how='left')
nsl = pd.DataFrame({
    'dataset':'nslkdd',
    'unit_id':'rung'+m['rung'].astype(str)+'_real'+m['realization'].astype(str),
    'seed':m['seed'],'arch':m['arch'],'protocol':m['protocol'],'class':m['class'],
    'is_focal':(m['class']=='R2L'),
    'n_eval':m['n_eval'].astype(int),'n_covered':m['n_covered'].astype(int),
    'S_cov':m['S_cov'],'S_lab':m['S_lab'],'S_sup':m['S_sup'],
    'alpha':ALPHA,'nominal':round(1-ALPHA,4)})
assert nsl['S_cov'].notna().all(), 'NSL shift merge left gaps'
print('NSL rows:', len(nsl), '| S_cov', round(nsl.S_cov.min(),3),'-',round(nsl.S_cov.max(),3),
      '| S_sup', round(nsl.S_sup.min(),3),'-',round(nsl.S_sup.max(),3))


NSL rows: 36000 | S_cov 0.84 - 0.903 | S_sup 0.0 - 0.457


In [4]:
# =============================================================================
# Cell 4 - CIC rows: recompute per-draw counts from cic_probs (Benign, DoS).
# Reconstruct Wednesday + realizations exactly as nb13, same draw seeds, emit
# n_eval/n_covered per class per draw. Focal = DoS. Attach per-realization shift.
# =============================================================================
cic = pd.read_parquet(config.INTERIM_DIR/'cicids2017_primary.parquet')
wed = cic[cic['day']=='wednesday'].reset_index(drop=True)
wed = wed[wed['label'].isin(['DoS','Benign'])].reset_index(drop=True)
CIC_PROBS = config.DATA_DIR/'cic_probs'
CIC_CLASSES=['Benign','DoS']; CIC_NC=2; CIC_FOCAL='DoS'
REALIZATIONS=['R1_holdout_Slowhttptest','R2_holdout_Slowloris','R3_holdout_GoldenEye',
              'R4_holdout_Slowloris_Slowhttptest','R5_holdout_GoldenEye_Slowloris']
cshift = pd.read_csv(config.REPORTS_DIR/'ladder_shift_measures_cicids2017.csv').set_index('realization')

def labels_cic(idx): return (wed.loc[idx,'label'].to_numpy()=='DoS').astype(int)
rows=[]
for name in REALIZATIONS:
    spx=np.load(config.PROC_DIR/f'cic_{name}_srcpool_idx.npy'); tgx=np.load(config.PROC_DIR/f'cic_{name}_target_idx.npy')
    ysp,ytg=labels_cic(spx),labels_cic(tgx); mm=min(len(spx),len(tgx)//2)
    Sc,Sl,Ss=float(cshift.loc[name,'S_cov']),float(cshift.loc[name,'S_lab']),float(cshift.loc[name,'S_sup'])
    for f in sorted(CIC_PROBS.glob(f'{name}__*.npz')):
        _,arch,sp=f.stem.split('__'); seed=int(sp.replace('seed',''))
        d=np.load(f); Psp,Ptg=d['srcpool'],d['target']
        for draw in range(R):
            rng=np.random.default_rng(dseed(name,seed,arch,draw))
            tp=rng.permutation(len(ytg)); de,tc=tp[:mm],tp[mm:2*mm]; sc=rng.permutation(len(ysp))[:mm]
            es=aps_scores_all(Ptg[de], np.random.default_rng(dseed(name,seed,arch,draw,'e')))
            cal={'REC':(aps_scores_all(Ptg[de],np.random.default_rng(dseed(name,seed,arch,draw,'rc'))),ytg[de]),
                 'TSC':(aps_scores_all(Ptg[tc],np.random.default_rng(dseed(name,seed,arch,draw,'tc'))),ytg[tc]),
                 'SHC':(aps_scores_all(Psp[sc],np.random.default_rng(dseed(name,seed,arch,draw,'sc'))),ysp[sc])}
            for proto,(cs,yc) in cal.items():
                ne,nc=cov_counts(es,ytg[de],cs,yc,ALPHA,CIC_NC)
                for c in range(CIC_NC):
                    rows.append(('cicids2017',f'{name}_draw{draw}',seed,arch,proto,CIC_CLASSES[c],
                                 CIC_CLASSES[c]==CIC_FOCAL,ne[c],nc[c],Sc,Sl,Ss,ALPHA,round(1-ALPHA,4)))
cicdf=pd.DataFrame(rows,columns=COLS)
print('CIC rows:', len(cicdf))


CIC rows: 9000


In [5]:
# =============================================================================
# Cell 5 - UGR16 rows: recompute per-draw counts from ugr16_probs (5 classes).
# Reconstruct source pool + target exactly as nb17, same draw seeds. Focal =
# nerisbotnet. Single S_cov/S_lab/S_sup for the whole environment.
# =============================================================================
UGR=config.DATASETS_DIR/'ugr16'
usrc=pd.read_parquet(UGR/'july_week5.parquet'); utgt=pd.read_parquet(UGR/'august_week1.parquet')
for dd in (usrc,utgt): dd['label']=dd['label'].astype(str).str.strip().str.lower()
UKEEP=['background','dos','scan11','scan44','nerisbotnet']
usrc=usrc[usrc.label.isin(UKEEP)].reset_index(drop=True); utgt=utgt[utgt.label.isin(UKEEP)].reset_index(drop=True)
UCLASSES=sorted(UKEEP); U2I={c:i for i,c in enumerate(UCLASSES)}; UGR_NC=len(UCLASSES); UGR_FOCAL='nerisbotnet'
def strat(df,fr,seed,col='label'):
    rng=np.random.default_rng(seed); nm=list(fr); f=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(f))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(f*n).astype(int); c[nm.index(big)]+=n-c.sum(); k=0
        for a2,q in zip(nm,c): a.loc[idx[k:k+q]]=a2; k+=q
    return a
usrc=usrc.assign(partition=strat(usrc,config.SPLIT_FRACTIONS,20260725).values)
y_sp=usrc[usrc.partition=='source_cal_pool']['label'].map(U2I).to_numpy()
y_tg=utgt['label'].map(U2I).to_numpy()
ush=json.load(open(config.REPORTS_DIR/'ladder_shift_measures_ugr16.json'))
USc,USl,USs=float(ush['S_cov']),float(ush['S_lab']),float(ush['S_sup'])
UGR_PROBS=config.DATA_DIR/'ugr16_probs'; mmU=min(len(y_sp),len(y_tg)//2)
rows=[]
for f in sorted(UGR_PROBS.glob('ugr16__*.npz')):
    _,arch,sp=f.stem.split('__'); seed=int(sp.replace('seed',''))
    d=np.load(f); Psp,Ptg=d['srcpool'],d['target']
    for draw in range(R):
        rng=np.random.default_rng(dseed('ugr16',seed,arch,draw))
        tp=rng.permutation(len(y_tg)); de,tc=tp[:mmU],tp[mmU:2*mmU]; sc=rng.permutation(len(y_sp))[:mmU]
        es=aps_scores_all(Ptg[de], np.random.default_rng(dseed('ugr16',seed,arch,draw,'e')))
        cal={'REC':(aps_scores_all(Ptg[de],np.random.default_rng(dseed('ugr16',seed,arch,draw,'r'))),y_tg[de]),
             'TSC':(aps_scores_all(Ptg[tc],np.random.default_rng(dseed('ugr16',seed,arch,draw,'t'))),y_tg[tc]),
             'SHC':(aps_scores_all(Psp[sc],np.random.default_rng(dseed('ugr16',seed,arch,draw,'s'))),y_sp[sc])}
        for proto,(cs,yc) in cal.items():
            ne,nc=cov_counts(es,y_tg[de],cs,yc,ALPHA,UGR_NC)
            for c in range(UGR_NC):
                rows.append(('ugr16',f'draw{draw}',seed,arch,proto,UCLASSES[c],
                             UCLASSES[c]==UGR_FOCAL,ne[c],nc[c],USc,USl,USs,ALPHA,round(1-ALPHA,4)))
ugrdf=pd.DataFrame(rows,columns=COLS)
print('UGR rows:', len(ugrdf))


UGR rows: 4500


In [ ]:
# =============================================================================
# Cell 6 - stack into the pooled table, report, commit.
# =============================================================================
pooled=pd.concat([nsl,cicdf,ugrdf],ignore_index=True)
pooled['coverage']=(pooled['n_covered']/pooled['n_eval'].replace(0,np.nan)).round(4)
pooled.to_csv(config.REPORTS_DIR/'pooled_coverage.csv',index=False)

print('POOLED TABLE:', pooled.shape)
print(pooled.groupby('dataset').agg(rows=('n_eval','size'),
      S_cov_min=('S_cov','min'),S_cov_max=('S_cov','max'),
      S_sup_min=('S_sup','min'),S_sup_max=('S_sup','max')).round(3).to_string())
print('\nfocal rows per dataset:', pooled[pooled.is_focal].groupby('dataset').size().to_dict())
print('\nSHC focal coverage by dataset (sanity vs per-dataset results):')
foc=pooled[(pooled.is_focal)&(pooled.protocol=='SHC')]
print((foc.groupby('dataset').apply(lambda g: g['n_covered'].sum()/g['n_eval'].sum())).round(4).to_string())

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb18: pooled cross-dataset coverage table (NSL counts + CIC/UGR recomputed with counts + shift covariates)')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


POOLED TABLE: (49500, 15)
             rows  S_cov_min  S_cov_max  S_sup_min  S_sup_max
dataset                                                      
cicids2017   9000      0.760      0.773      0.011      0.070
nslkdd      36000      0.840      0.903      0.000      0.457
ugr16        4500      0.692      0.692      0.000      0.000

focal rows per dataset: {'cicids2017': 4500, 'nslkdd': 9000, 'ugr16': 900}

SHC focal coverage by dataset (sanity vs per-dataset results):
dataset
cicids2017    0.5899
nslkdd        0.0860
ugr16         0.9473


/tmp/ipykernel_658/2448130502.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print((foc.groupby('dataset').apply(lambda g: g['n_covered'].sum()/g['n_eval'].sum())).round(4).to_string())
